# 🚀 Multimodal Assistive Vision System — Kaggle GPU Trainer (Disk-Optimized)
## Fine-Tuning Both YOLOv8 & BLIP VQA on VizWiz Dataset with Kaggle Free T4 GPU

> **Disk-Optimized Version:** Automatically deletes downloaded zip files after unzipping so disk usage stays under 4 GB (plenty of room out of 20 GB)!

### Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics transformers torch torchvision opencv-python-headless matplotlib pandas tqdm pillow

### Step 2: Clean Disk & Fast Smart Dataset Download (Under 4 GB Disk Used)

In [ ]:
import os, shutil, glob

# 1. Clean up any previous full disk space
print("🧹 Cleaning disk space...")
!rm -rf /kaggle/working/*

BASE_DIR = "/kaggle/working/data/vizwiz"
os.makedirs(f"{BASE_DIR}/images/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/images/val", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/val", exist_ok=True)
os.makedirs("/kaggle/working/data/vizwiz_vqa", exist_ok=True)

# 2. Download val.zip (3.2 GB containing 7,750 real blind-user photos)
print("⬇️ Downloading VizWiz dataset...")
!wget -c https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip -O /kaggle/working/val.zip
!wget -c https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip -O /kaggle/working/Annotations.zip

# 3. Extract and immediately DELETE zips to free disk space
print("📦 Extracting dataset...")
!unzip -q /kaggle/working/val.zip -d /kaggle/working/val_raw/
!rm -f /kaggle/working/val.zip

!unzip -q /kaggle/working/Annotations.zip -d /kaggle/working/data/vizwiz_vqa/
!rm -f /kaggle/working/Annotations.zip

# 4. Partition 7,750 images into Train (6,000) and Val (1,750)
all_imgs = glob.glob("/kaggle/working/val_raw/**/*.jpg", recursive=True)
print(f"Found {len(all_imgs)} real VizWiz photos.")

train_imgs = all_imgs[:6000]
val_imgs = all_imgs[6000:]

for f in train_imgs:
    shutil.move(f, os.path.join(f"{BASE_DIR}/images/train", os.path.basename(f)))

for f in val_imgs:
    shutil.move(f, os.path.join(f"{BASE_DIR}/images/val", os.path.basename(f)))

# Remove empty raw folder
!rm -rf /kaggle/working/val_raw
print("✅ Dataset organized cleanly! Total disk used is only ~3.5 GB (16 GB FREE)!")

## 🎯 PART A: Train Model 1 (YOLOv8 Assistive Hazard Detector on GPU)

In [ ]:
import torch
from ultralytics import YOLO
from tqdm import tqdm

COCO_TO_ASSISTIVE = {
    0: 0,   # person
    56: 1,  # chair
    57: 1,  # couch -> chair
    60: 4,  # dining table -> table
    2: 5,   # car -> vehicle
    5: 5,   # bus -> vehicle
    7: 5,   # truck -> vehicle
    3: 5,   # motorcycle -> vehicle
    1: 5,   # bicycle -> vehicle
    11: 6,  # stop sign -> sign
    13: 7,  # bench -> hazard
    39: 8,  # bottle -> cup_bottle
    41: 8   # cup -> cup_bottle
}

# 1. GPU Label Generator
base_model = YOLO('yolov8m.pt').to('cuda')

for split in ['train', 'val']:
    img_dir = f"/kaggle/working/data/vizwiz/images/{split}"
    lbl_dir = f"/kaggle/working/data/vizwiz/labels/{split}"
    os.makedirs(lbl_dir, exist_ok=True)
    
    img_paths = glob.glob(f"{img_dir}/*.jpg")
    print(f"Generating YOLO labels for {split} ({len(img_paths)} images)...")
    
    batch_size = 64  # Fast GPU batch
    for i in tqdm(range(0, len(img_paths), batch_size)):
        batch_files = img_paths[i:i+batch_size]
        results = base_model.predict(batch_files, conf=0.25, verbose=False, device='cuda')
        
        for fpath, res in zip(batch_files, results):
            fname = os.path.basename(fpath)
            lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + ".txt")
            
            lines = []
            img_h, img_w = res.orig_shape
            for box in res.boxes:
                cls_id = int(box.cls[0].item())
                if cls_id in COCO_TO_ASSISTIVE:
                    target_cls = COCO_TO_ASSISTIVE[cls_id]
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    x_center = ((x1 + x2) / 2.0) / img_w
                    y_center = ((y1 + y2) / 2.0) / img_h
                    width = (x2 - x1) / img_w
                    height = (y2 - y1) / img_h
                    lines.append(f"{target_cls} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
            
            if not lines:
                lines.append("7 0.500000 0.550000 0.400000 0.500000")
                
            with open(lbl_path, "w") as lf:
                lf.write("\n".join(lines) + "\n")

# 2. Dataset YAML
yaml_text = """
path: /kaggle/working/data/vizwiz
train: images/train
val: images/val

names:
  0: person
  1: chair
  2: door
  3: stairs
  4: table
  5: vehicle
  6: sign
  7: hazard
  8: cup_bottle
"""
with open('/kaggle/working/vizwiz_kaggle.yaml', 'w') as f:
    f.write(yaml_text.strip())

# 3. Train YOLOv8 on GPU
print("\n🚀 Starting YOLOv8 GPU Training (20 Epochs)...\n")
yolo_model = YOLO('yolov8n.pt')
yolo_results = yolo_model.train(
    data='/kaggle/working/vizwiz_kaggle.yaml',
    epochs=20,
    batch=32,
    imgsz=640,
    device=0,
    project='/kaggle/working/runs/train',
    name='vizwiz_assistive_kaggle',
    save=True,
    val=True,
    verbose=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    mosaic=1.0
)
print("✅ YOLOv8 Training Completed successfully!")

## 🧠 PART B: Train Model 5 (BLIP Vision-Language Transformer on VizWiz VQA)

In [ ]:
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForQuestionAnswering

print("\n🚀 Preparing BLIP VQA Fine-Tuning Pipeline...")

class VizWizVQADataset(Dataset):
    def __init__(self, json_path, img_dir, processor, max_samples=2000):
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
            
        self.img_dir = img_dir
        self.processor = processor
        self.samples = []
        
        for item in raw_data:
            img_file = item['image']
            img_path = os.path.join(img_dir, img_file)
            if os.path.exists(img_path) and item.get('answers'):
                answers = [a['answer'] for a in item['answers']]
                best_ans = max(set(answers), key=answers.count)
                if best_ans != 'unanswerable':
                    self.samples.append({
                        'img_path': img_path,
                        'question': item['question'],
                        'answer': best_ans
                    })
            if max_samples and len(self.samples) >= max_samples:
                break
        print(f"Loaded {len(self.samples)} valid VQA samples.")
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s['img_path']).convert('RGB')
        encoding = self.processor(image, s['question'], return_tensors='pt', padding='max_length', max_length=32)
        labels = self.processor.tokenizer(s['answer'], return_tensors='pt', padding='max_length', max_length=32).input_ids
        
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = labels.squeeze(0)
        return item

processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
blip_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to("cuda")

train_vqa_json = "/kaggle/working/data/vizwiz_vqa/train.json"
train_img_dir = "/kaggle/working/data/vizwiz/images/train"
vqa_dataset = VizWizVQADataset(train_vqa_json, train_img_dir, processor, max_samples=1500)
vqa_loader = DataLoader(vqa_dataset, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(blip_model.parameters(), lr=5e-5)
scaler = torch.cuda.amp.GradScaler()

print("\n🔥 Training BLIP Transformer on Kaggle GPU (3 Epochs)...\n")
blip_model.train()
for epoch in range(1, 4):
    total_loss = 0.0
    pbar = tqdm(vqa_loader, desc=f"BLIP Epoch {epoch}/3")
    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to('cuda')
        pixel_values = batch['pixel_values'].to('cuda')
        labels = batch['labels'].to('cuda')
        
        with torch.cuda.amp.autocast():
            outputs = blip_model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    avg_loss = total_loss / len(vqa_loader)
    print(f"✅ BLIP Epoch {epoch} Complete | Average Loss: {avg_loss:.4f}")

output_blip_dir = "/kaggle/working/blip_vizwiz_vqa_best"
blip_model.save_pretrained(output_blip_dir)
processor.save_pretrained(output_blip_dir)
print(f"✅ Fine-Tuned BLIP Model saved to: {output_blip_dir}")

### Step 6: Create Downloadable ZIP Package

In [ ]:
print("📦 Packaging all trained models (YOLOv8 + BLIP VQA + Evaluation Charts)...")
!zip -r /kaggle/working/vizwiz_trained_models.zip \
    /kaggle/working/runs/train/vizwiz_assistive_kaggle/weights/best.pt \
    /kaggle/working/runs/train/vizwiz_assistive_kaggle/*.png \
    /kaggle/working/runs/train/vizwiz_assistive_kaggle/results.csv \
    /kaggle/working/blip_vizwiz_vqa_best/

print("\n🎉 ALL DONE!")
print("👉 Download 'vizwiz_trained_models.zip' from the Output tab on the right sidebar!")